> Adapted from *Dive into Deep Learning*, section [Parameter Management](https://d2l.ai/chapter_builders-guide/parameters.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 6.1: Layers and Modules](06.01_model-construction.ipynb) | Next: [Section 6.3: Parameter Initialization](06.03_init-param.ipynb)

# Parameter Management

Once we have chosen an architecture
and set our hyperparameters,
we proceed to the training loop,
where our goal is to find parameter values
that minimize our loss function.
After training, we will need these parameters
in order to make future predictions.
Additionally, we will sometimes wish
to extract the parameters
perhaps to reuse them in some other context,
to save our model to disk so that
it may be executed in other software,
or for examination in the hope of
gaining scientific understanding.

Most of the time, we will be able
to ignore the nitty-gritty details
of how parameters are declared
and manipulated, relying on deep learning frameworks
to do the heavy lifting.
However, when we move away from
stacked architectures with standard layers,
we will sometimes need to get into the weeds
of declaring and manipulating parameters.
In this section, we cover the following:

* Accessing parameters for debugging, diagnostics, and visualizations.
* Sharing parameters across different model components.

In [ ]:
import numpy as np
from divelet_ml import nn

import divelet_ml as d2l

We start by focusing on an MLP with one hidden layer.

In [ ]:
net = nn.Sequential(nn.LazyLinear(8),
                    nn.ReLU(),
                    nn.LazyLinear(1))

X = np.random.rand(2, 4)
net(X).shape

## Parameter Access

Let's start with how to access parameters
from the models that you already know.

When a model is defined via the `Sequential` class,
we can first access any layer by indexing
into the model as though it were a list.
Each layer's parameters are conveniently
located in its attribute.


We can inspect the parameters of the second fully connected layer as follows.

In [ ]:
net[2].state_dict()

We can see that this fully connected layer
contains two parameters,
corresponding to that layer's
weights and biases, respectively.


### Targeted Parameters

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> Note that each parameter is represented
as a NumPy array, held as an attribute of its layer.
The following code extracts the bias
from the second neural network layer.

In [ ]:
type(net[2].bias), net[2].bias

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> A parameter is a plain array: it holds its values only, with no gradient and no other information.
The gradients come from autograd, as in [Section 3.4](https://d2l.ai/chapter_linear-regression/linear-regression-scratch.html#sec-linear-regression-scratch):
`d2l.param_grads` differentiates a function with respect to the parameter arrays of a model,
and the optimizer's `step(grads)` updates them in place.

### All Parameters at Once

When we need to perform operations on all parameters,
accessing them one-by-one can grow tedious.
The situation can grow especially unwieldy
when we work with more complex, e.g., nested, modules,
since we would need to recurse
through the entire tree to extract
each sub-module's parameters. Below we demonstrate accessing the parameters of all layers.

In [ ]:
[(name, param.shape) for name, param in net.named_parameters()]

## Tied Parameters

Often, we want to share parameters across multiple layers.
Let's see how to do this elegantly.
In the following we allocate a fully connected layer
and then use its parameters specifically
to set those of another layer.
Here we need to run the forward propagation
`net(X)` before accessing the parameters.

In [ ]:
# We need to give the shared layer a name so that we can refer to its
# parameters
shared = nn.LazyLinear(8)
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.LazyLinear(1))

net(X)
# Check whether the parameters are the same
print(net[2].weight[0] == net[4].weight[0])
net[2].weight[0, 0] = 100
# Make sure that they are actually the same object rather than just having the
# same value
print(net[2].weight[0] == net[4].weight[0])

This example shows that the parameters
of the second and third layer are tied.
They are not just equal, they are
represented by the same exact tensor.
Thus, if we change one of the parameters,
the other one changes, too.

You might wonder,
when parameters are tied
what happens to the gradients?
Since autograd differentiates with respect to the one array that both layers use,
the gradients from the second hidden layer
and the third hidden layer are added together
during backpropagation.


## Summary

We have several ways of accessing and tying model parameters.


## Exercises

1. Use the `NestMLP` model defined in [Section 6.1](https://d2l.ai/chapter_builders-guide/model-construction.html#sec-model-construction) and access the parameters of the various layers.
1. Construct an MLP containing a shared parameter layer and train it. During the training process, observe the model parameters and gradients of each layer.
1. Why is sharing parameters a good idea?



Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/57)

Previous: [Section 6.1: Layers and Modules](06.01_model-construction.ipynb) | Next: [Section 6.3: Parameter Initialization](06.03_init-param.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[Parameter Management](https://d2l.ai/chapter_builders-guide/parameters.html)
(source: [`chapter_builders-guide/parameters.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_builders-guide/parameters.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text and figures: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python). The word *tensor* is kept and means a NumPy `ndarray`.
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`.
- A parameter is a NumPy array held as an attribute of its layer, so `.data` is dropped (`net[2].bias` is the array itself) and there is no `.grad`: the paragraph that shows the gradient in its initial state says instead that gradients come from autograd, and its code cell is removed.
- `state_dict()` and `named_parameters()` are methods of divelet-ml's `nn.Module`; `named_parameters()` lists a shared array once, as PyTorch does. `torch.rand` is `np.random.rand`.
- The sentence on tied parameters' gradients says that autograd adds the gradients of the two uses, where the book says that the parameters contain gradients.
- Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.